## Baseline and Evaluation Protocol

In this part we set up the evaluation protocol shared by every model, written in `utils.py`, and evaluate the baseline: a model that always predicts the mean of the train folds. Every model of the next notebooks must beat it.

### Protocol (`utils.py`)

- **Rows:** One regression per target, trained on the rows where this target is known (`get_xy`). `charpy_temp_C` is an input of the Charpy model only.
- **Inputs:** The 30 inputs of the EDA notebook (chemistry, welding parameters, PWHT and welding process), standardised inside a `Pipeline` (`build_pipeline`: `StandardScaler` + model), so that the scaler is fitted on the train folds only.
- **Folds:** 5 folds grouped on `weld_group` and stratified on the alloy family (`StratifiedGroupKFold`, seed 42). `load_train` gives each deposit its fold once for all, so the folds are the same for every target and every person, and every fold holds Cr-Mo welds.
- **Regression metrics:** MAE and RMSE of the out-of-fold predictions, with their standard deviation over the 5 folds. The MAE is in the unit of the target and easy to read, the RMSE penalises the large errors more.
- **Conformity:** The thresholds of `definition_bonne_soudure.md` are applied to the out-of-fold predictions, with the thresholds of the family of each row. For Charpy, the model predicts the energy at the reference temperature of the family (-40 °C for C-Mn, +20 °C for Cr-Mo), by changing `charpy_temp_C` in its input.
- **Classification metrics:** F1 and recall, with "non-conforming" as the positive class. They are computed per criterion on the rows where it is decided, and for the full label on the labelled weld deposits. The recall is the share of bad welds the model catches, which is what matters most to a manufacturer. The F1 also penalises the false alarms.
- **Results table:** `results.csv`, one row per model, target and family (`all`, `C-Mn`, `CrMo2`, `CrMo91`). `save_results` replaces the previous rows of the same model.

### How to evaluate a model

The model notebooks go in this folder, next to `utils.py`, so that `from utils import ...` works. The hyperparameters are tuned per target by a grid search on the same folds, then `evaluate` cross-validates the best model of each target:

```python
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from utils import SEED, TARGETS, load_train, get_xy, build_pipeline, evaluate, save_results

train = load_train()

best = {}
for target in TARGETS:
    X, y, cv = get_xy(train, target)
    search = GridSearchCV(
        build_pipeline(RandomForestRegressor(random_state=SEED)),
        {"model__max_depth": [5, 10, None]},
        cv=cv,
        scoring="neg_mean_absolute_error",
    )
    search.fit(X, y)
    best[target] = search.best_estimator_

save_results(evaluate("Random forest", best, train))
```

### Operations in order

1. Load the train set with `load_train`.
2. Check the folds: each deposit in a single fold, deposits per fold and family, rows per fold and target.
3. Count the decided and non-conforming rows of each criterion.
4. Count the labelled deposits, and compute the F1 of a rule that declares every weld non-conforming.
5. Evaluate the baseline, a `DummyRegressor` that predicts the mean of the train folds.
6. Save it to `results.csv`.

In [1]:
import pandas as pd
from sklearn.dummy import DummyRegressor

from utils import TARGETS, load_train, get_xy, true_criteria, deposit_labels, evaluate, save_results

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

In [3]:
train = load_train()

print(train.shape)
print("Deposits in more than one fold:", (train.groupby("weld_group")["fold"].nunique() > 1).sum())

deposits = train.drop_duplicates("weld_group")
pd.crosstab(deposits["family"], deposits["fold"], margins=True)

(1229, 40)
Deposits in more than one fold: 0


fold,0,1,2,3,4,All
family,,,,,,
C-Mn,85,83,85,84,83,420
CrMo2,6,6,7,6,7,32
CrMo9,0,0,0,1,0,1
CrMo91,5,7,6,6,6,30
All,96,96,98,97,96,483


- **fold:** Each deposit is in a single fold, and each fold holds 96 to 98 deposits, of which 6 or 7 CrMo2 and 5 to 7 CrMo91. The single CrMo9 deposit is in fold 3.

In [4]:
pd.DataFrame({target: train.loc[get_xy(train, target)[0].index, "fold"].value_counts().sort_index() for target in TARGETS})

,yield_strength_MPa,uts_MPa,elongation_pct,charpy_toughness_J
fold,,,,
0,117,116,104,147
1,116,112,101,141
2,131,127,125,141
3,130,127,115,136
4,118,109,104,139


The folds are fixed on the deposits, not on each target, so the number of rows per fold varies a little from one target to another (from 101 to 147 rows).

In [5]:
criteria = true_criteria(train)

pd.DataFrame({
    "decided": criteria.notna().sum(),
    "non_conforming": (criteria == 0).sum(),
    "non_conforming_pct": ((criteria == 0).sum() / criteria.notna().sum() * 100).round(1),
})

,decided,non_conforming,non_conforming_pct
yield_strength_MPa,611,102,16.7
uts_MPa,590,187,31.7
elongation_pct,548,49,8.9
charpy_toughness_J,296,30,10.1


- **yield_strength_MPa, uts_MPa, elongation_pct:** Decided on every row where they are measured, except the Cr-Mo row without PWHT, which is outside the conditions of its standard. The UTS fails most often (32% of the rows), on C-Mn welds outside the 500-640 MPa range.
- **charpy_toughness_J:** Only 296 of the 704 Charpy rows are decided. The others were tested at a temperature that does not allow a conclusion (see `definition_bonne_soudure.md`).

In [6]:
labels = deposit_labels(train, criteria, criteria)

summary = labels.groupby("family")["true_nc"].agg(["size", "sum"])
summary.loc["all"] = [len(labels), labels["true_nc"].sum()]
summary.columns = ["deposits", "non_conforming"]
display(summary)

p = labels["true_nc"].mean()
print("Share of non-conforming deposits:", round(p, 3))
print("F1 of 'every weld is non-conforming':", round(2 * p / (1 + p), 3))

,deposits,non_conforming
family,,
C-Mn,156,71
CrMo2,12,2
CrMo91,4,2
all,172,75


Share of non-conforming deposits: 0.436
F1 of 'every weld is non-conforming': 0.607


`deposit_labels` takes the measured criteria twice here, as the true and as the predicted criteria: only the true label `true_nc` is used.

- **Labelled deposits:** 172, the same count as `definition_bonne_soudure.md`, of which 75 are non-conforming. Only 16 are Cr-Mo, so the label metrics of the Cr-Mo families are only indicative.
- **Reference F1:** A rule that declares every weld non-conforming has a recall of 1 and an F1 of 2p / (1 + p), with p the share of non-conforming deposits, so 0.61 here. On the label, a model is only useful if its F1 is above this value.

In [7]:
baseline = evaluate("Baseline (mean)", DummyRegressor(strategy="mean"), train)
baseline.round(2)

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,Baseline (mean),yield_strength_MPa,all,612,71.80,93.13,8.54,11.47,611,102,0.0,0.0
1,Baseline (mean),yield_strength_MPa,C-Mn,550,66.89,83.61,5.30,7.97,550,100,0.0,0.0
2,Baseline (mean),yield_strength_MPa,CrMo2,29,79.70,106.98,27.92,36.49,28,2,0.0,0.0
3,Baseline (mean),yield_strength_MPa,CrMo91,32,146.50,185.93,48.07,50.69,32,0,0.0,0.0
4,Baseline (mean),uts_MPa,all,591,70.94,89.56,6.74,12.73,590,187,0.0,0.0
5,Baseline (mean),uts_MPa,C-Mn,533,65.97,80.80,4.81,6.09,533,187,0.0,0.0
6,Baseline (mean),uts_MPa,CrMo2,33,84.70,113.67,35.05,46.48,32,0,0.0,0.0
7,Baseline (mean),uts_MPa,CrMo91,25,158.69,182.67,43.22,61.54,25,0,0.0,0.0
8,Baseline (mean),elongation_pct,all,549,4.09,4.95,0.26,0.25,548,49,0.0,0.0
9,Baseline (mean),elongation_pct,C-Mn,490,3.83,4.63,0.19,0.17,490,37,0.0,0.0


- **MAE:** 72 MPa on the yield strength, 71 MPa on the UTS, 4.1% on the elongation and 39 J on the Charpy energy: the error of a model that knows nothing about the weld. It is about twice as large on the CrMo91 welds (146 and 159 MPa on the yield strength and the UTS), which are far from the mean of the train set.
- **F1, recall:** 0 everywhere. The mean of each target passes the thresholds, so the baseline declares every weld conforming and does not catch a single bad weld. Its accuracy on the label would still be 56% (97 conforming deposits out of 172), which is why the accuracy is not used.

In [8]:
results = save_results(baseline)
results[results["family"] == "all"].round(2)

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,Baseline (mean),yield_strength_MPa,all,612,71.80,93.13,8.54,11.47,611,102,0.0,0.0
4,Baseline (mean),uts_MPa,all,591,70.94,89.56,6.74,12.73,590,187,0.0,0.0
8,Baseline (mean),elongation_pct,all,549,4.09,4.95,0.26,0.25,548,49,0.0,0.0
12,Baseline (mean),charpy_toughness_J,all,704,39.17,49.25,2.27,3.28,296,30,0.0,0.0
16,Baseline (mean),label,all,<NA>,NaN,NaN,NaN,NaN,172,75,0.0,0.0


`results.csv` now holds the baseline. Each model notebook adds its rows with `save_results`, and the models are compared with `load_results()`, for example `load_results().pivot_table(index=["target", "family"], columns="model", values="MAE", sort=False)`.